# Tutorial 04: surface hopping with SHARC 4

*Author: Rhyan Barrett*

The CH₂S model from tutorial 02 in SHARC 4: single point, gas-phase and QM/MM (SHARC_QMMM + OpenMM)
trajectories with pysharc or `sharc.x`, compared with the LVC reference. Details: [README](README.md).

In [ ]:
import os, subprocess, sys
import numpy as np
from IPython.display import Image, display

QUICK = os.environ.get("FIELDMACE_TUTORIAL_QUICK") == "1"   # tiny settings for automatic checks
os.environ.setdefault("OMP_NUM_THREADS", "4")

def run(cmd, env=None):
    """Run a tutorial step and show its output (stops the notebook if the step fails)."""
    print("$", cmd)
    p = subprocess.run(cmd, shell=True, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                       env={**os.environ, **(env or {})})
    lines = [l for l in p.stdout.splitlines() if "Warning" not in l and "warnings.warn" not in l]
    print("\n".join(lines[-60:]))
    if p.returncode:
        raise RuntimeError(f"step failed: {cmd}")
assert os.environ.get("SHARC"), "set $SHARC (SHARC 4 bin directory) before starting Jupyter"

## Step 0: link the interface into SHARC and check the installation

In [ ]:
run("bash 00_install.sh")

## Step 1: one QM.in through SHARC_FIELDMACE.py and SHARC_LVC.py

In [ ]:
run("python 01_single_point.py")

In [ ]:
print(open("singlepoint/FIELDMACE/QM.out").read()[:2500])

## Steps 2–4: trajectories (gas phase), FieldMACE and LVC, same initial conditions

8 trajectories of 200 fs, starting in T2 (SHARC MCH state 4), which decays to T1 and mixes with S1.

In [ ]:
nt, tmax = (2, 2) if QUICK else (8, 200)
for iface in ("FIELDMACE", "LVC"):
    run(f"python 02_setup_trajectories.py --mode gas --interface {iface} --ntraj {nt} --tmax {tmax}")
    run(f"bash 03_run_trajectories.sh runs/gas_{iface}")
run("python 04_analyze.py runs/gas_FIELDMACE runs/gas_LVC")
display(Image("results/populations.png"))

## QM/MM: SHARC_QMMM with FieldMACE + OpenMM

`qm-program FIELDMACE`, `mm-program openmm`; `nophases_from_interface` in the SHARC input (SHARC_QMMM does
not pass phases on).

In [ ]:
nt, tmax = (1, 1) if QUICK else (4, 100)
for iface in ("FIELDMACE", "LVC"):
    run(f"python 02_setup_trajectories.py --mode qmmm --interface {iface} --ntraj {nt} --tmax {tmax}")
    run(f"bash 03_run_trajectories.sh runs/qmmm_{iface}")
run("python 04_analyze.py runs/qmmm_FIELDMACE runs/qmmm_LVC")
display(Image("results/populations.png"))

In [ ]:
print(open("runs/qmmm_FIELDMACE/TRAJ_00001/QM/QMMM.template").read())
print(open("runs/qmmm_FIELDMACE/TRAJ_00001/input").read())